# CS224N · Lecture 11 — Benchmarking and Evaluation

**Slides:** [cs224n-spr2024-lecture11-evaluation-yann.pdf](https://web.stanford.edu/class/cs224n/slides/cs224n-spr2024-lecture11-evaluation-yann.pdf) (Yann Dubois)  
**Prerequisites:** none beyond basic probability; BLEU from L7 and Bradley–Terry from L10 are re-implemented here.

> **Benchmarks and evaluations drive progress.** Whatever the field measures is what the field improves (MMLU is a recent example). So *how* we measure matters enormously, and evaluation is full of traps. This notebook turns each of the lecture's warnings into a small, runnable measurement.

| § | Topic | You will implement / measure |
|---|---|---|
| 1 | Why we evaluate: different desiderata at each stage | — |
| 2 | **Close-ended evaluation**: metrics, aggregation, spurious correlations | P/R/F1, macro vs. micro, AUC; aggregation flips rankings; a hypothesis-only baseline |
| 3 | **Open-ended evaluation**: content overlap, model-based metrics, references | ROUGE-1/2/L, the "Heck no!" failure, BERTScore's matching rule |
| 4 | **Human evaluation** and its issues | Cohen's and Fleiss' κ |
| 5 | **Chatbots**: side-by-side comparisons, Arena, LLM judges | Elo vs. Bradley–Terry; **length-controlled win rate**; position bias |
| 6 | Current LLM evaluation: perplexity, "everything" benchmarks, code, agents | Tokenizer-dependence of perplexity; unbiased **pass@k** |
| 7 | Issues: consistency, **contamination**, overfitting, monoculture, bias | MMLU scoring variants; **Min-K% prob** contamination detection |
| 8 | 🔎 Statistical significance | Bootstrap CIs, paired bootstrap, McNemar, sample-size planning |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random, math, collections, itertools
from scipy.stats import spearmanr, binomtest

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)
random.seed(0)

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

---
## 1. Different desiderata for measuring performance

The same model is evaluated at many stages, and each stage needs different things from its evaluation:

| Stage | What the evaluation must be |
|---|---|
| **Train** (the loss) | super fast, super cheap, **differentiable**, no shortcut for the model to exploit |
| **Develop** (dev set, while iterating) | super fast, super cheap, avoid shortcuts |
| **Model selection** | fast, cheap, **trustworthy**, task-specific |
| **Publish** (papers, leaderboards) | **absolute**, **standardized**, **reproducible**, easy to work with; ~fast, ~cheap; broad coverage; **fine-grained distinguishability**; good difficulty (not saturated) |
| **Deploy** | task-specific; crude metrics may be fine |

Two major types of evaluations:
* **Close-ended:** a limited number of potential answers, often one or a few correct; enables **automatic evaluation as in ML**.
* **Open-ended:** long generations with too many correct answers to enumerate; there are now **better and worse** answers, not just right and wrong.

---
## 2. Close-ended evaluation

Classic tasks: **sentiment analysis** (SST, IMDB, Yelp), **entailment** (SNLI), **named entity recognition** (CoNLL-2003), **part-of-speech** (PTB), **coreference resolution** (WSC), **question answering** (SQuAD 2). **Multi-task benchmarks** like **SuperGLUE** attempt to measure "general language capabilities": BoolQ and MultiRC (reading texts), CB and RTE (entailment), COPA (cause and effect), ReCoRD (QA + reasoning), WiC (meaning of words), WSC (coreference).

**Close-ended challenges:** choosing your metrics (accuracy / precision / recall / F1 / ROC); aggregating across metrics or tasks; where do the labels come from? Are there spurious correlations?

### 2.1 Choosing your metric
For a class $c$: **precision** = (predicted $c$ and truly $c$) / (predicted $c$); **recall** = (predicted $c$ and truly $c$) / (truly $c$); $F_1$ = their harmonic mean. Over classes, **macro**-averaging averages the per-class scores (every class counts equally); **micro**-averaging pools all decisions (frequent classes dominate; for single-label tasks micro-F1 = accuracy). **ROC-AUC** measures ranking quality across all thresholds: the probability that a random positive gets a higher score than a random negative.

In [ ]:
def per_class_prf(y_true, y_pred, classes):
    out = {}
    for c in classes:
        tp = np.sum((y_pred == c) & (y_true == c)); fp = np.sum((y_pred == c) & (y_true != c)); fn = np.sum((y_pred != c) & (y_true == c))
        p = tp / (tp + fp) if tp + fp else 0.0; r_ = tp / (tp + fn) if tp + fn else 0.0
        out[c] = (p, r_, 2 * p * r_ / (p + r_) if p + r_ else 0.0)
    return out

def summary(y_true, y_pred, classes):
    prf = per_class_prf(y_true, y_pred, classes)
    return {"accuracy": np.mean(y_true == y_pred), "macro-F1": np.mean([v[2] for v in prf.values()]),
            "F1 of rare class": prf[classes[-1]][2]}

def roc_auc(scores, labels):
    # P(score of random positive > score of random negative), ties count 1/2 (Mann-Whitney U)
    pos, neg = scores[labels == 1], scores[labels == 0]
    return (np.mean(pos[:, None] > neg[None, :]) + 0.5 * np.mean(pos[:, None] == neg[None, :]))

r = np.random.default_rng(0)
y = (r.random(1000) < 0.05).astype(int)                          # a rare positive class (5%), e.g. "toxic"
always_negative = np.zeros_like(y)
noisy_detector_scores = y * 1.0 + r.normal(0, 0.6, len(y))
detector = (noisy_detector_scores > 0.5).astype(int)
for name, pred in [("always predict negative", always_negative), ("a real (noisy) detector", detector)]:
    print(f"{name:26s}", {k: round(float(v), 3) for k, v in summary(y, pred, [0, 1]).items()})
print(f"detector ROC-AUC = {roc_auc(noisy_detector_scores, y):.3f}   (trivial classifier: 0.5)")

A useless classifier gets **95% accuracy** on a 5%-positive task. Accuracy hides it; macro-F1 and the rare class's F1 expose it. Pick the metric that reflects what you actually care about.

### 2.2 Aggregating across tasks
Benchmarks like SuperGLUE report one number, but *how* you aggregate is a choice, and it can change the ranking:

In [ ]:
tasks = ["BoolQ", "CB", "COPA", "RTE", "WiC", "WSC"]
scores = {"system A": np.array([0.90, 0.95, 0.92, 0.88, 0.40, 0.89]),       # strong but fails one task badly
          "system B": np.array([0.80, 0.82, 0.81, 0.79, 0.78, 0.80])}       # consistently decent
rank_of = lambda vals: {k: i + 1 for i, k in enumerate(sorted(vals, key=lambda k: -vals[k]))}
aggs = {"arithmetic mean": lambda s: s.mean(), "geometric mean": lambda s: np.exp(np.log(s).mean()),
        "worst task (min)": lambda s: s.min(), "median": lambda s: np.median(s)}
for agg_name, f in aggs.items():
    vals = {k: f(v) for k, v in scores.items()}
    print(f"{agg_name:18s}: " + "   ".join(f"{k} = {v:.3f}" for k, v in vals.items()) + f"   -> winner: {max(vals, key=vals.get)}")

### 2.3 Spurious correlations
SNLI itself is hard, but there can be **undiscovered spurious correlations**. Gururangan et al. (2018) found that crowdworkers writing *contradiction* hypotheses often just added a **negation** (*Premise: The economy could be still better. Hypothesis: The economy has **never** been better.*). A model can then score well **without even reading the premise**.

Let's build an NLI-like dataset with that artifact, and measure a **hypothesis-only** classifier, a model that cannot possibly be doing inference:

In [ ]:
SUBJ = ["the economy", "the team", "the weather", "the movie", "the city", "the market"]
PRED = ["is good", "is better", "is improving", "is strong", "is busy", "is calm"]
NEG = ["never", "not"]
LABELS = ["entailment", "neutral", "contradiction"]

def nli_example(r, artifact_strength):
    label = r.integers(3); s = SUBJ[r.integers(len(SUBJ))]; p_ = PRED[r.integers(len(PRED))]
    premise = f"{s} {p_}"
    use_neg = r.random() < (artifact_strength if label == 2 else (1 - artifact_strength) / 2)
    hyp = f"{s} {NEG[r.integers(2)]} {p_}" if use_neg else f"{s} {PRED[r.integers(len(PRED))]}"
    return premise, hyp, label

def bow(texts, vocab):
    X = np.zeros((len(texts), len(vocab)))
    for i, t in enumerate(texts):
        for w in t.split():
            if w in vocab:
                X[i, vocab[w]] = 1
    return X

def train_softmax(X, y, n_classes=3, steps=500, lr=0.5):
    W = np.zeros((X.shape[1], n_classes))
    for _ in range(steps):
        P = np.exp(X @ W); P /= P.sum(1, keepdims=True)
        P[np.arange(len(y)), y] -= 1; W -= lr * X.T @ P / len(y)
    return W

r = np.random.default_rng(0)
train = [nli_example(r, 0.8) for _ in range(3000)]
biased_test = [nli_example(r, 0.8) for _ in range(1000)]          # same artifact as training (a standard test split)
stress_test = [nli_example(r, 1 / 3) for _ in range(1000)]        # negation equally common in every class
vocab = {w: i for i, w in enumerate(sorted({w for _, h, _ in train for w in h.split()}))}
W = train_softmax(bow([h for _, h, _ in train], vocab), np.array([l for *_, l in train]))
for name, data in [("standard test split", biased_test), ("stress test (artifact removed)", stress_test)]:
    pred = (bow([h for _, h, _ in data], vocab) @ W).argmax(1)
    print(f"hypothesis-only classifier on {name:32s}: accuracy {np.mean(pred == np.array([l for *_, l in data])):.1%}   (chance 33.3%)")

The hypothesis-only model looks far above chance on the standard split, while knowing nothing about entailment, and drops to chance once the artifact is removed. **Always run "partial-input" baselines** (hypothesis-only, question-only, image-only...): if they do well, your benchmark measures something other than what you think.

---
## 3. Open-ended evaluation

**Open-ended tasks:** long generations with too many possible correct answers to enumerate, so we can't use standard ML metrics. Examples: summarization (CNN-DM, Gigaword), translation (WMT), instruction following (Chatbot Arena, AlpacaEval, MT-Bench).

Types of evaluation methods for text generation: **content overlap metrics**, **model-based metrics**, and **human evaluations**.

### 3.1 Content overlap metrics
Compute a score that indicates the **lexical similarity** between generated and gold-standard (human-written) text. Fast and efficient. N-gram overlap metrics (BLEU, ROUGE, METEOR, CIDEr) are **not ideal, but often still reported** for translation and summarization. *Ref: They walked to the grocery store. Gen: The woman went to the hardware store.* BLEU is precision-oriented (L7); **ROUGE** (Lin 2004) is recall-oriented, designed for summarization:
* **ROUGE-N:** n-gram recall (and precision/F1) against the reference
* **ROUGE-L:** based on the **longest common subsequence** (LCS), rewarding in-order matches without requiring them to be contiguous

In [ ]:
def ngrams(toks, n):
    return collections.Counter(tuple(toks[i:i + n]) for i in range(len(toks) - n + 1))

def rouge_n(cand, ref, n):
    c, r_ = ngrams(cand, n), ngrams(ref, n)
    overlap = sum((c & r_).values())
    p = overlap / max(sum(c.values()), 1); rec = overlap / max(sum(r_.values()), 1)
    return p, rec, (2 * p * rec / (p + rec) if p + rec else 0.0)

def lcs_len(a, b):
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a)):
        for j in range(len(b)):
            dp[i + 1][j + 1] = dp[i][j] + 1 if a[i] == b[j] else max(dp[i][j + 1], dp[i + 1][j])
    return dp[-1][-1]

def rouge_l(cand, ref):
    l = lcs_len(cand, ref); p, rec = l / len(cand), l / len(ref)
    return p, rec, (2 * p * rec / (p + rec) if p + rec else 0.0)

ref = "they walked to the grocery store .".split()
gen = "the woman went to the hardware store .".split()
for name, f in [("ROUGE-1", lambda c, r_: rouge_n(c, r_, 1)), ("ROUGE-2", lambda c, r_: rouge_n(c, r_, 2)), ("ROUGE-L", rouge_l)]:
    p, rec, f1 = f(gen, ref)
    print(f"{name}: precision {p:.3f}  recall {rec:.3f}  F1 {f1:.3f}")

**A simple failure case: n-gram overlap metrics have no concept of semantic relatedness!** The slides' example: the question *"Are you enjoying the CS224N lectures?"* with reference answer *"Heck yes !"*

In [ ]:
ref = "heck yes !".split()
cands = ["you know it !", "yes !", "yup .", "heck no !"]
print(f"{'candidate':16s} {'unigram precision':>18s} {'unigram recall':>15s} {'F1':>6s}   meaning")
for c, meaning in zip(cands, ["agrees", "agrees", "agrees", "DISAGREES"]):
    p, rec, f1 = rouge_n(c.split(), ref, 1)
    print(f"{c:16s} {p:18.2f} {rec:15.2f} {f1:6.2f}   {meaning}")

*"Yup."* (a perfect answer) scores 0, a **false negative**. *"Heck no!"* (the opposite meaning) scores as well as *"Yes!"*, a **false positive**.

### 3.2 Model-based metrics: capture more semantics
Use **learned representations** of words and sentences to compute semantic similarity between generated and reference texts. The embeddings are pretrained; the distance metric can be fixed.
* **Word distance functions / vector similarity:** embedding average, vector extrema (Liu et al. 2016), MEANT, YiSi.
* **BERTScore** (Zhang et al. 2020): uses pretrained **contextual** embeddings from BERT and matches words in candidate and reference by cosine similarity: each candidate token is matched to its most similar reference token (precision), each reference token to its most similar candidate token (recall), then $F_1$.
* **BLEURT** (Sellam et al. 2020): a regression model based on BERT that returns a score for how grammatical the candidate is and how well it conveys the reference's meaning, trained on human ratings.

Below is BERTScore's matching rule. To keep it self-contained, the token vectors here are **hand-made toy embeddings** with three interpretable dimensions (affirmative, negative, punctuation), standing in for BERT's contextual vectors; the optional cell in §9 runs the real thing.

In [ ]:
toy_emb = {"heck": [0.3, 0.3, 0], "yes": [1, 0, 0], "yup": [0.95, 0, 0.05], "no": [0, 1, 0], "know": [0.6, 0, 0.2],
           "you": [0.2, 0.1, 0.3], "it": [0.2, 0.1, 0.3], "!": [0, 0, 1], ".": [0, 0, 1]}
def embed(toks):
    E = np.array([toy_emb[t] for t in toks], float)
    return E / np.linalg.norm(E, axis=1, keepdims=True)

def bertscore(cand, ref):
    C, R = embed(cand), embed(ref)
    sim = C @ R.T                                         # cosine similarity of every candidate/reference token pair
    p = sim.max(1).mean(); rec = sim.max(0).mean()        # greedy matching
    return p, rec, 2 * p * rec / (p + rec)

for c in cands:
    print(f"{c:16s} BERTScore-style F1 = {bertscore(c.split(), ref)[2]:.3f}")

With embeddings that encode meaning, *"yup ."* now scores well. But notice *"heck no !"* still scores fairly high: it shares two exact tokens with the reference, and embedding-matching metrics are still **lenient about negation**, a known weakness. Model-based metrics are better, not solved.

### 3.3 An important failure case: references
**Reference-based measures are only as good as their references.** For summarization, scores computed against the dataset's actual references (often noisy, e.g., news "highlights" never meant as summaries) are nearly **uncorrelated** with human judgments, while the same metrics against **expert-written** references correlate much better (Zhang et al. 2023, on benchmarking LLMs for news summarization).

**Reference-free evaluation:** have a model give a score, with no human reference. This was nonstandard, and became popular with GPT-4 (AlpacaEval, MT-Bench; §5).

---
## 4. Human evaluation

Automatic metrics fall short of matching human decisions, so **human evaluation is the most important form of evaluation for text generation**, and the **gold standard** for developing new automatic metrics: new metrics must correlate well with human evaluations!

Ask humans to evaluate generated text **overall** or along specific dimensions: **fluency**, **coherence/consistency**, **factuality and correctness**, **commonsense**, **style/formality**, **grammaticality**, **redundancy**. *Don't compare human evaluation scores across differently conducted studies, even if they claim to evaluate the same dimensions!*

**Human evaluation has issues too:** slow; expensive; **inter-annotator disagreement** (especially if subjective); **intra-annotator disagreement** across time; **not reproducible** (Belz et al. 2023: *"just 5% of human evaluations are repeatable in the sense that (i) there are no prohibitive barriers to repetition, and (ii) sufficient information about experimental design is publicly available for rerunning them"*; about 20% with author help); measures **precision, not recall** (humans judge what they see, not what's missing); and **biases/shortcuts if incentives aren't aligned** (annotators maximizing $/hour). Designing one is hard: how to describe the task, how to show it, what metric, selecting and monitoring annotators (time, accuracy).

### 4.1 Measuring agreement: chance-corrected κ
Raw percent agreement is misleading when one label dominates: two annotators who *both* say "fine" 90% of the time agree ~82% by chance alone. **Cohen's κ** (two annotators) and **Fleiss' κ** (many) correct for chance:
$$\kappa = \frac{p_o - p_e}{1 - p_e}\qquad(p_o = \text{observed agreement},\ p_e = \text{agreement expected by chance})$$

In [ ]:
def cohens_kappa(a, b):
    labels = np.union1d(a, b)
    po = np.mean(a == b)
    pe = sum(np.mean(a == l) * np.mean(b == l) for l in labels)
    return (po - pe) / (1 - pe)

def fleiss_kappa(ratings, n_labels):
    # ratings: (items, annotators) integer labels
    n_items, n_ann = ratings.shape
    counts = np.stack([(ratings == l).sum(1) for l in range(n_labels)], 1)          # (items, labels)
    P_i = ((counts * (counts - 1)).sum(1)) / (n_ann * (n_ann - 1))
    p_j = counts.sum(0) / (n_items * n_ann)
    return (P_i.mean() - (p_j ** 2).sum()) / (1 - (p_j ** 2).sum())

r = np.random.default_rng(0)
truth = (r.random(500) < 0.1).astype(int)                     # 10% of outputs are actually bad
def annotator(skill):                                         # with prob. skill, report the truth; else guess "fine" 90% of the time
    guess = (r.random(500) < 0.1).astype(int)
    return np.where(r.random(500) < skill, truth, guess)
for skill in [0.0, 0.5, 0.9]:
    A = np.stack([annotator(skill) for _ in range(3)], 1)
    print(f"annotator skill {skill:.1f}: raw agreement (annotators 1,2) {np.mean(A[:, 0] == A[:, 1]):.1%}   "
          f"Cohen's kappa {cohens_kappa(A[:, 0], A[:, 1]):.3f}   Fleiss' kappa (3 annotators) {fleiss_kappa(A, 2):.3f}")

Annotators who are **guessing at random** (skill 0) still show ~82% raw agreement, because both mostly say "fine"; κ correctly reports ~0. Rough conventions: κ < 0.2 slight, 0.2–0.4 fair, 0.4–0.6 moderate, 0.6–0.8 substantial, > 0.8 almost perfect (Landis & Koch 1977), but what's "good enough" depends on the task's inherent subjectivity.

---
## 5. Reference-free evaluation of chatbots

How do we evaluate something like ChatGPT? There are so many different use cases that it's hard to evaluate, and the responses are long-form text, which is even harder.

### 5.1 Side-by-side ratings: Chatbot Arena
Have people play with **two (anonymous) models side by side** and say which response is better. This is the **current gold standard** for evaluating chat LLMs. Votes are turned into a leaderboard with a rating system. LMSYS originally used **Elo** (from chess); they moved to fitting the **Bradley–Terry** model (L10) by maximum likelihood, because online Elo updates depend on the **order** in which battles happen, and LLM ratings shouldn't. Let's see why:

In [ ]:
r = np.random.default_rng(0)
models = ["model-A", "model-B", "model-C", "model-D", "model-E", "model-F"]
true_strength = np.array([1.2, 0.9, 0.5, 0.4, 0.0, -0.6])            # Bradley-Terry strengths (natural log-odds units)
battles = []
for _ in range(3000):
    i, j = r.choice(len(models), 2, replace=False)
    battles.append((i, j) if r.random() < sigmoid(true_strength[i] - true_strength[j]) else (j, i))
battles = np.array(battles)                                          # (winner, loser)

def elo(battles, k=16, init=1000):
    R = np.full(len(models), float(init))
    for w, l in battles:
        e_w = 1 / (1 + 10 ** ((R[l] - R[w]) / 400))                  # expected score of the winner
        R[w] += k * (1 - e_w); R[l] -= k * (1 - e_w)
    return R

def bradley_terry(battles, n, steps=2000, lr=1.0):
    s = np.zeros(n)
    for _ in range(steps):
        g_ = 1 - sigmoid(s[battles[:, 0]] - s[battles[:, 1]])
        grad = np.zeros(n); np.add.at(grad, battles[:, 0], g_); np.add.at(grad, battles[:, 1], -g_)
        s += lr * grad / len(battles)
    return 1000 + (s - s.mean()) * 400 / np.log(10)                  # report on the Elo scale

elo_runs = np.array([elo(battles[np.random.default_rng(seed).permutation(len(battles))]) for seed in range(20)])
bt = bradley_terry(battles, len(models))
boot = np.array([bradley_terry(battles[np.random.default_rng(100 + b_).integers(0, len(battles), len(battles))], len(models), steps=500)
                 for b_ in range(30)])
print(f"{'model':8s} {'true':>6s} {'Elo range over 20 battle orders':>32s} {'Bradley-Terry':>14s} {'BT 95% CI':>16s}")
for k, name in enumerate(models):
    lo, hi = np.percentile(boot[:, k], [2.5, 97.5])
    print(f"{name:8s} {1000 + (true_strength[k] - true_strength.mean()) * 400 / np.log(10):6.0f} "
          f"{elo_runs[:, k].min():14.0f} to {elo_runs[:, k].max():6.0f}           {bt[k]:6.0f}      [{lo:.0f}, {hi:.0f}]")

The **same 3,000 votes** give different Elo ratings depending only on their order, enough to swap closely matched models. Bradley–Terry uses all votes at once and is order-independent; bootstrapping the battles gives confidence intervals (Arena reports these; overlapping intervals mean "statistically tied").

**What's missing with side-by-side human eval?**
* **External validity:** typing random questions into a head-to-head website may not be representative of real use.
* **Cost:** human annotation takes a large community effort; new models take a long time to benchmark; only notable models get benchmarked.

### 5.2 Lowering the costs: use an LM as the evaluator
Use an LM as a **reference-free evaluator** (an "LLM judge"). Surprisingly high correlations with humans. Common versions: **AlpacaEval**, **MT-Bench**. In AlpacaFarm, the LLM judge was **~100× cheaper, ~100× faster**, and had **higher agreement with the human majority vote than individual humans did**, because humans have low agreement due to variance. (LLM judges can also generate preference labels for training: RLAIF, L10.)

**AlpacaEval:** an internal benchmark for developing Alpaca; **98% correlation with Chatbot Arena**; takes **< 3 min and < \$10**:
1. For each instruction, generate an output with a baseline model and with the model to evaluate.
2. Ask GPT-4 the probability that the model's output is better.
3. (**AlpacaEval LC**) Reweight the win probability based on the **length** of the outputs.
4. Average the win probabilities → **win rate**.

**Things to be careful with: the same issues as before, spurious correlations!**
* **Length:** judges (and humans) prefer longer answers.
* **Position:** judges prefer the answer shown first (or second); everyone randomizes this away.
* **Self-bias:** GPT-4 favors GPT-4 outputs, though surprisingly not by much.

### 5.3 Length-controlled win rate
The question AlpacaEval LC asks: **what would the win rate be if the baseline and model outputs had the same length?** Fit a logistic regression of the judge's preference on (a model term) + (a length-difference term), then predict with the length difference set to zero (Dubois et al. 2024). We simulate a judge with a length bias and two models: one genuinely better, one that just writes longer answers.

In [ ]:
r = np.random.default_rng(0)
n = 800
def judge_prefers_model(quality_gap, len_model, len_base, length_bias=0.8):
    # simulated LLM judge: depends on true quality AND (spuriously) on length difference (in hundreds of tokens)
    z = quality_gap + length_bias * np.tanh((len_model - len_base) / 300)
    return (r.random(len(z)) < sigmoid(z)).astype(float)

len_base = r.normal(300, 80, n)
systems = {"better model, same length": (0.6, len_base + r.normal(0, 80, n)),
           "same quality, 2.5x longer": (0.0, 2.5 * len_base + r.normal(0, 80, n))}

def lc_win_rate(wins, len_m, len_b, steps=3000, lr=0.5):
    feat = np.tanh((len_m - len_b) / 300)                      # standardized length difference
    a, b = 0.0, 0.0                                            # logit = a (model) + b * length difference
    for _ in range(steps):
        p = sigmoid(a + b * feat)
        a += lr * np.mean(wins - p); b += lr * np.mean((wins - p) * feat)
    return sigmoid(a)                                          # predicted win rate when length difference = 0

for name, (gap, len_m) in systems.items():
    wins = judge_prefers_model(np.full(n, gap), len_m, len_base)
    print(f"{name:28s} raw win rate {wins.mean():.1%}   length-controlled win rate {lc_win_rate(wins, len_m, len_base):.1%}"
          f"   (true quality-only win rate {sigmoid(gap):.1%})")

The verbose model's raw win rate is inflated well above 50% with **no** quality advantage; the length-controlled estimate removes most of that inflation, while the genuinely better model keeps its advantage. (This regression approach assumes the length effect is captured by the modeled term; it's a correction, not a guarantee.)

**Position bias** is handled by design rather than modeling: evaluate each pair **twice with the order swapped**, and count a win only if it holds in both orders (or average the two).

In [ ]:
def biased_judge(prefers_A_prob, position_bias=0.25):
    # returns P(judge picks the response shown FIRST); first-position responses get a boost
    return np.clip(prefers_A_prob + position_bias, 0, 1)

p_true = 0.5                                            # two equally good responses
shown_A_first = biased_judge(p_true)                    # P(A wins) when A is first
shown_B_first = 1 - biased_judge(1 - p_true)            # P(A wins) when B is first
print(f"P(A wins) if always shown first: {shown_A_first:.2f};  averaged over both orders: {(shown_A_first + shown_B_first) / 2:.2f} (correct: 0.50)")

---
## 6. Current evaluation of LLMs

How LLMs are evaluated depends on the stage: **perplexity** (pretraining), **"everything" benchmarks** (finetuned models), and **arena-like** comparisons.

### 6.1 Perplexity
Perplexity is **highly correlated with downstream performance**, but it **depends on the data and the tokenizer**. Models with different tokenizers produce different numbers of tokens for the same text, so per-token perplexities aren't comparable. Normalize by something tokenizer-independent: **bits per byte** (or per character) of the original text.

In [ ]:
text = ("language models assign probabilities to text . the probability of a text is the product of the "
        "probabilities of its tokens . a model that predicts text well has low perplexity . ") * 20
def unigram_bits(tokens):
    counts = collections.Counter(tokens); total = sum(counts.values())
    return -sum(math.log2(counts[t] / total) for t in tokens)            # total bits to encode the text

for name, toks in [("characters", list(text)), ("words", text.split())]:
    bits = unigram_bits(toks)
    print(f"unigram model over {name:10s}: {len(toks):5d} tokens, per-token perplexity {2 ** (bits / len(toks)):7.2f}, "
          f"bits per character {bits / len(text):.3f}")

The word-level model has a *higher* per-token perplexity (it chooses among many more possible tokens at each step), yet it needs about **5× fewer bits** for the whole text: it's the better model of this text. Per-token perplexity rewards splitting text into many easy tokens; bits per character compares models fairly. (These are unigram models trained and evaluated on the same text, purely to show the normalization issue.)

### 6.2 "Everything" benchmarks: HELM and the Open LLM Leaderboard
**Holistic Evaluation of Language Models (HELM)** and the **Hugging Face Open LLM Leaderboard** collect many automatically evaluatable benchmarks and evaluate models across all of them. What do these benchmarks evaluate on? A huge mix: knowledge (MMLU, the 57-subject multiple-choice exam), reasoning (ARC, HellaSwag, BIG-Bench), math (GSM8K), truthfulness (TruthfulQA), and more.

### 6.3 Code: HumanEval and pass@k
A nice feature of code: **evaluate against test cases** (functional correctness, not text overlap). **HumanEval** (Chen et al. 2021): 164 hand-written programming problems with unit tests. The metric is **pass@k**: the probability that at least one of $k$ sampled solutions passes. (GPT-4: ~67% pass@1 on the original report.)

The naive estimate $1 - (1 - \hat p)^k$ is **biased**. Chen et al.'s **unbiased estimator**: generate $n\geq k$ samples, count the $c$ that pass, and compute
$$\text{pass@}k = 1 - \frac{\binom{n-c}{k}}{\binom{n}{k}}$$
the probability that a random size-$k$ subset of the $n$ samples contains at least one correct one.

In [ ]:
def pass_at_k(n, c, k):
    if n - c < k:
        return 1.0
    return 1.0 - np.prod(1.0 - k / np.arange(n - c + 1, n + 1))       # = 1 - C(n-c, k)/C(n, k), numerically stable

r = np.random.default_rng(0)
true_p = r.beta(0.5, 2.0, 164)                                        # per-problem success rate of a model (164 problems)
k, n = 10, 20
truth = np.mean(1 - (1 - true_p) ** k)                                # true pass@10
unbiased, naive = [], []
for trial in range(300):
    c = r.binomial(n, true_p)
    unbiased.append(np.mean([pass_at_k(n, ci, k) for ci in c]))
    naive.append(np.mean(1 - (1 - c / n) ** k))
print(f"true pass@{k}: {truth:.4f}")
print(f"unbiased estimator (n={n}): mean {np.mean(unbiased):.4f}   naive 1-(1-c/n)^k: mean {np.mean(naive):.4f}")

### 6.4 Agents
LMs are increasingly used for more than text, e.g., actuating **agents** (browsing, using tools, editing code). **Challenge: evaluation needs to be done in sandbox environments** (the agent must actually act, and its actions must be checked safely). Lecture 14 covers agents.

---
## 7. Issues and challenges with evaluation

### 7.1 Consistency issues
Small changes to how a benchmark is run change the scores, sometimes the rankings (Alzahrani et al. 2024 show leaderboard rankings shifting from minor formatting changes). **MMLU has many implementations:** different prompts, and different ways to score the answer: (a) the probability of the answer **letter** "A/B/C/D"; (b) the probability of the **full answer text**; (c) **generate** and parse; (d) restrict to the most likely **valid** choice. These can disagree on the same model:

In [ ]:
# A worked example: one question, one model, four scoring rules (log-probs are illustrative numbers).
choices = ["A", "B", "C", "D"]
answer_text = ["Paris", "Lyon", "the capital is Paris, France", "Marseille"]
logp_letter = np.log([0.30, 0.40, 0.20, 0.10])                              # P(" A"), P(" B"), ... after the prompt
logp_text_per_token = [np.log([0.6]), np.log([0.3]), np.log([0.9, 0.9, 0.9, 0.9, 0.9, 0.8]), np.log([0.2])]
generated = "The answer is Paris"
rules = {
    "letter probability": choices[int(np.argmax(logp_letter))],
    "full-text log-prob (sum)": choices[int(np.argmax([lp.sum() for lp in logp_text_per_token]))],
    "full-text log-prob (per token)": choices[int(np.argmax([lp.mean() for lp in logp_text_per_token]))],
    "generate, then parse": next((c for c, t in zip(choices, answer_text) if t in generated), "unparseable"),
}
for rule, pick in rules.items():
    print(f"{rule:32s} -> {pick}")

Four reasonable implementations, three different answers. Summed log-probs penalize long answers, per-token averages favor them, letter probabilities test whether the model knows the *format*, and parsing depends on the regex. **Report exactly how you scored**, and compare models only under the same protocol.

### 7.2 Contamination and overfitting issues
**Contamination:** with closed models and web-scale pretraining, it's **hard to know whether benchmarks are truly "new"**: the test questions may be in the training data. **Overfitting:** benchmarks reach "human-level" performance too quickly (saturation), and the community overfits to the test sets through repeated use (L7's dev-set demo).

**Alleviating overfitting:** control the number of times one can see the test set (a **private test set**); or **constantly change the inputs** (a **dynamic test set**, e.g., Dynabench, or LiveBench's regularly refreshed questions); Arena-style evaluation is naturally dynamic.

**Alleviating contamination: detectors.**
* **Min-K% prob** (Shi et al. 2023): text seen in training rarely contains very *surprising* tokens. Average the log-probabilities of the $k\%$ least likely tokens; "too high" suggests the text was seen. (What is too high? Often heuristic.)
* **Exchangeability test** (Oren et al. 2023): look for specific signatures, like a preference for the benchmark's canonical **ordering** of examples, that can only be learned by peeking at the dataset.

Let's implement Min-K% prob. We train a trigram LM on a "web corpus" into which **half of a benchmark's questions leaked**, then try to detect which ones:

In [ ]:
r = np.random.default_rng(0)
WORDS = [f"w{i}" for i in range(400)]                                   # a 400-word vocabulary
def random_text(n):
    return [WORDS[i] for i in r.zipf(1.3, n) % len(WORDS)]               # Zipfian word frequencies
benchmark = [random_text(15) for _ in range(200)]
leaked = set(range(0, 200, 2))                                            # every other benchmark question leaked
web = [random_text(15) for _ in range(4000)] + [benchmark[i] for i in leaked]

counts3, counts2, counts1 = collections.Counter(), collections.Counter(), collections.Counter()
for s in web:
    t = ["<s>", "<s>"] + s
    for i in range(2, len(t)):
        counts3[tuple(t[i - 2:i + 1])] += 1; counts2[tuple(t[i - 2:i])] += 1; counts1[t[i]] += 1
total1 = sum(counts1.values())
def token_logprobs(s):
    t = ["<s>", "<s>"] + s; out = []
    for i in range(2, len(t)):
        p3 = counts3[tuple(t[i - 2:i + 1])] / counts2[tuple(t[i - 2:i])] if counts2[tuple(t[i - 2:i])] else 0
        p1 = (counts1[t[i]] + 1) / (total1 + len(WORDS))
        out.append(math.log(0.8 * p3 + 0.2 * p1))                        # interpolated trigram
    return np.array(out)

def min_k_prob(s, k=0.2):
    lp = np.sort(token_logprobs(s))
    return lp[:max(1, int(len(lp) * k))].mean()                           # mean of the k% LOWEST log-probs

scores = np.array([min_k_prob(s) for s in benchmark]); labels = np.array([i in leaked for i in range(200)]).astype(int)
avg_lp = np.array([token_logprobs(s).mean() for s in benchmark])
print(f"contamination detection AUC: Min-20% prob {roc_auc(scores, labels):.3f}   average log-prob {roc_auc(avg_lp, labels):.3f}")

Both detectors separate leaked from clean questions in this idealized setting, where the leaked text appears **verbatim** in the training data. (Here even the plain average log-prob works. Min-K%'s advantage in Shi et al. shows up with real LMs, where a text's average is dominated by many easy, common tokens, while its rarest tokens carry the memorization signal.) Real detection is much harder: paraphrased or translated contamination, partial overlap, and the lack of a trustworthy threshold without known-clean reference data. This is why dynamic and private test sets matter.

### 7.3 Monoculture of NLP benchmarking
**Most papers only evaluate on English and on performance (accuracy).** **Multilingual benchmarks exist; we should use them!** MEGA (16 datasets, 70 languages), GlobalBench (966 datasets in 190 languages), XTREME (9 tasks, 40 languages), and MMLU/ARC/HellaSwag translated into 26 languages.

### 7.4 The reductive single-metric issue
Performance is not all we care about:
* **Computational efficiency:** **MLPerf** measures the *time to achieve a desired quality target*.
* **Biases:** **DiscrimEval** (Anthropic) is template-based: how would the model's decision change based on the group mentioned (same scenario, different demographic attributes)?
* **Taking averages for aggregation is unfair to minoritized groups:** a high average can hide a subgroup with very poor performance (like the "worst task" aggregation in §2.2, applied to people).
* **Different preferences for different people:** there's no single "right" ranking of chatbot answers.

**Other biases in our evaluations:**
* **Biased metrics:** n-gram overlap metrics (BLEU, ROUGE) are not suited to languages with rich morphology or unclear tokenization.
* **Biased LLM-based evaluations:** an LLM judge's preferences are likely representative of only a small subgroup of people.
* **Opinions and values:** OpinionQA / GlobalOpinionQA (Santurkar et al. 2023; Durmus et al. 2023) compare LMs' answer distributions to public opinion surveys: *whose opinions do LLMs reflect by default?* (Instruction-tuned models shift toward particular demographics; L10.)

### 7.5 The challenge of challenges: the status quo
Academic researchers are incentivized to keep using the same benchmark to compare to previous work. **82% of machine translation papers in 2019–2020 evaluated only on BLEU** (Marie et al. 2021), despite many metrics that correlate better with human judgment.

---
## 8. 🔎 Is the difference real? Statistical significance

*(Not in the slides, but essential for anyone comparing models.)* A test set is a sample. With 500 examples, accuracy has a standard error of about $\sqrt{0.8\times0.2/500}\approx1.8$ points, so "82.0 vs. 81.2" may mean nothing.

* **Bootstrap confidence interval:** resample the test set with replacement many times, recompute the metric, take percentiles.
* **Paired comparisons:** two systems evaluated on the **same** examples are correlated; compare them *per example*. The **paired bootstrap** resamples examples and asks how often system B beats A. **McNemar's test** uses only the examples where the two systems disagree.

In [ ]:
r = np.random.default_rng(0)
n = 500
difficulty = r.normal(size=n)                                          # shared example difficulty -> correlated systems
sys_A = (r.random(n) < sigmoid(1.6 - difficulty)).astype(int)
sys_B = (r.random(n) < sigmoid(1.75 - difficulty)).astype(int)          # truly slightly better
print(f"accuracy A = {sys_A.mean():.3f}, B = {sys_B.mean():.3f}, difference = {sys_B.mean() - sys_A.mean():+.3f}")

boot = r.integers(0, n, (5000, n))
ci_A = np.percentile(sys_A[boot].mean(1), [2.5, 97.5]); ci_B = np.percentile(sys_B[boot].mean(1), [2.5, 97.5])
diff = sys_B[boot].mean(1) - sys_A[boot].mean(1)                     # paired: the same resampled examples for both
boot2 = r.integers(0, n, (5000, n))
diff_unpaired = sys_B[boot2].mean(1) - sys_A[boot].mean(1)            # unpaired: independent resamples
print(f"unpaired bootstrap: 95% CI of B - A = [{np.percentile(diff_unpaired, 2.5):+.3f}, {np.percentile(diff_unpaired, 97.5):+.3f}]")
print(f"95% bootstrap CI: A [{ci_A[0]:.3f}, {ci_A[1]:.3f}]   B [{ci_B[0]:.3f}, {ci_B[1]:.3f}]   (they overlap)")
print(f"paired bootstrap: 95% CI of B - A = [{np.percentile(diff, 2.5):+.3f}, {np.percentile(diff, 97.5):+.3f}];  "
      f"P(B <= A) under resampling = {np.mean(diff <= 0):.3f}")
b01, b10 = np.sum((sys_A == 0) & (sys_B == 1)), np.sum((sys_A == 1) & (sys_B == 0))
print(f"McNemar: B right & A wrong on {b01} examples, A right & B wrong on {b10};  exact p-value = {binomtest(b01, b01 + b10).pvalue:.3f}")

def n_needed(p, delta, power_z=0.84, alpha_z=1.96):
    # approximate test-set size to detect an accuracy difference delta between two INDEPENDENT evaluations
    return int(np.ceil(2 * p * (1 - p) * (alpha_z + power_z) ** 2 / delta ** 2))
for delta in [0.05, 0.02, 0.01]:
    print(f"to detect a {delta:.0%} accuracy difference at ~80% accuracy (80% power, unpaired): ~{n_needed(0.8, delta):,} examples")

**System B really is better** (we built it that way), and it scores 2 points higher, yet **no test can confirm it with 500 examples**: the per-system intervals overlap, the paired bootstrap interval includes 0, and McNemar's p-value is far above 0.05. Pairing helps a little here (the paired interval is slightly narrower than the unpaired one, because both systems find the same examples hard); it helps much more when two systems' errors are strongly correlated, as with two checkpoints of the same model. But a 2-point gain at this test-set size is simply within noise. The sample-size table explains why: detecting a 1-point difference takes tens of thousands of examples. **Many leaderboard gaps and "state-of-the-art" claims are within noise.** Report confidence intervals, and use paired tests.

### 8.1 🔎 Beyond the slides: calibration, i.e. can you trust the model's confidence?
Accuracy says how often a model is right; **calibration** says whether its stated confidence means anything. A model is calibrated if, among predictions made with confidence 0.8, about 80% are correct. This matters whenever a confidence drives a decision: abstaining, routing to a human, thresholding a classifier, or deciding whether to trust an LLM's answer.
* **Expected calibration error (ECE):** bin predictions by confidence; ECE $=\sum_b\frac{n_b}{n}\,|\text{acc}(b) - \text{conf}(b)|$. The **reliability diagram** plots accuracy vs. confidence per bin.
* Modern neural networks are often **overconfident** (Guo et al. 2017). For LLMs, the GPT-4 technical report found the pretrained model well calibrated on MMLU, and the RLHF'd model noticeably less so.
* **Temperature scaling** is a simple, strong fix: divide the logits by one scalar $T$, fit on a held-out set to minimize NLL. It never changes the argmax (accuracy is unchanged), only the confidence.

In [ ]:
r = np.random.default_rng(0)
def make_split(n):
    true_logits = r.normal(0, 1.5, (n, 4)); y = np.array([r.choice(4, p=np.exp(l) / np.exp(l).sum()) for l in true_logits])
    return true_logits * 3.0, y                                      # the model's logits are 3x too sharp: overconfident
logits_dev, y_dev = make_split(3000); logits_test, y_test = make_split(3000)

def softmax_rows(z):
    e = np.exp(z - z.max(1, keepdims=True)); return e / e.sum(1, keepdims=True)

def ece(probs, y, n_bins=10):
    conf, pred = probs.max(1), probs.argmax(1); bins = np.minimum((conf * n_bins).astype(int), n_bins - 1)
    out, diag = 0.0, []
    for b_ in range(n_bins):
        m = bins == b_
        if m.any():
            out += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean()); diag.append((conf[m].mean(), (pred[m] == y[m]).mean()))
    return out, np.array(diag)

Ts = np.linspace(0.5, 6, 200)
nll = [-np.log(softmax_rows(logits_dev / T)[np.arange(len(y_dev)), y_dev]).mean() for T in Ts]
T_best = Ts[int(np.argmin(nll))]
before, d_before = ece(softmax_rows(logits_test), y_test); after, d_after = ece(softmax_rows(logits_test / T_best), y_test)
acc_b, acc_a = (softmax_rows(logits_test).argmax(1) == y_test).mean(), (softmax_rows(logits_test / T_best).argmax(1) == y_test).mean()
print(f"fitted temperature on dev: T = {T_best:.2f} (true miscalibration factor: 3.0)")
print(f"test ECE: before {before:.3f} -> after {after:.3f};   accuracy before {acc_b:.3f}, after {acc_a:.3f} (unchanged)")
plt.figure(figsize=(4.5, 4)); plt.plot([0, 1], [0, 1], "k:", label="perfect calibration")
plt.plot(d_before[:, 0], d_before[:, 1], "o-", label="before"); plt.plot(d_after[:, 0], d_after[:, 1], "s-", label=f"temperature-scaled (T={T_best:.1f})")
plt.xlabel("confidence"); plt.ylabel("accuracy"); plt.legend(fontsize=8); plt.title("Reliability diagram"); plt.show()

### 8.2 🔎 Beyond the slides: evaluating a deployed system
Benchmarks are *offline* evaluation. Once a system is deployed, practitioners also rely on:
* **Online evaluation / A/B tests:** randomly route a fraction of real traffic to the new model and compare business or user metrics (task success, thumbs-up rate, retention, latency, cost). This is the only evaluation that measures real users, but it's slow and needs careful statistics (§8: the same significance ideas apply).
* **Slice-based evaluation:** report metrics per **slice** (language, user segment, input length, topic, demographic group); an average can hide a slice that got worse (§7.4).
* **Regression test suites:** a fixed set of important cases (including past bugs) that every new model must pass, like unit tests for models. For LLM apps, these are often scored by assertions or an LLM judge.
* **Monitoring for drift:** input distributions change over time; track input statistics and output quality signals in production.

---
## 9. Optional: real BERTScore

The real BERTScore with contextual embeddings from a pretrained model (here `roberta-large` via the `bert-score` package, or any encoder via `transformers`).

In [ ]:
# [Optional] needs: pip install bert-score  (downloads roberta-large, ~1.4GB)
try:
    from bert_score import score
    cands_ = ["You know it!", "Yes!", "Yup.", "Heck no!"]
    P, R, F = score(cands_, ["Heck yes!"] * len(cands_), lang="en", verbose=False)
    for c, f in zip(cands_, F.tolist()):
        print(f"{c:14s} BERTScore F1 = {f:.4f}")
except Exception as e:
    print("Skipping BERTScore demo:", e)

---
## 10. Evaluation: takeaways

* **Close-ended tasks:** think about what you evaluate (diversity, difficulty), and choose metrics and aggregation deliberately.
* **Open-ended tasks:** content overlap metrics are useful in low-diversity settings (e.g., translation); **chatbot evals are very difficult**, and selecting the right examples/evaluation is an open problem.
* **Challenges:** **consistency** (hard to know if we're evaluating the right thing), **contamination** (can we trust the numbers?), **biases**.
* **In many cases, the best judge of output quality is YOU! Look at your model generations. Don't just rely on numbers!**

| Concept | Takeaway |
|---|---|
| Metric choice | Accuracy hides rare-class failure; use macro-F1 / per-class / AUC as appropriate |
| Aggregation | Mean vs. geometric mean vs. worst-case can flip rankings |
| Spurious correlations | Run partial-input baselines (e.g., hypothesis-only) |
| Overlap metrics | No semantics: false negatives ("Yup.") and false positives ("Heck no!") |
| Model-based metrics | BERTScore/BLEURT: better, still lenient on negation; only as good as references |
| Human eval | Gold standard, but noisy; report chance-corrected agreement (κ) |
| Arena | Pairwise votes → Bradley–Terry (order-independent) with bootstrap CIs |
| LLM judges | Cheap and correlated with humans; control length, randomize position, beware self-bias |
| Perplexity | Tokenizer-dependent; compare in bits per byte/character |
| pass@k | Use the unbiased estimator $1 - \binom{n-c}{k}/\binom{n}{k}$ |
| Contamination | Min-K% prob etc. help in easy cases; prefer private/dynamic tests |
| Significance | Paired tests (paired bootstrap, McNemar); small gaps need big test sets |

### Exercises
1. In §2.3, add a premise–hypothesis **word-overlap** feature to the classifier. Does it help on the stress test?
2. Implement **ROUGE-L with multiple references** (take the max over references) and **chrF** (character n-gram F-score).
3. In §5.1, make the battles **non-uniform**: the top two models battle each other 10× more often. How do Elo and Bradley–Terry react?
4. In §5.3, make the length effect **nonlinear** in a way the regression doesn't model (e.g., only lengths above 600 tokens are favored). Does LC still remove the bias?
5. In §7.2, **paraphrase** leaked questions (randomly replace 30% of their words) before adding them to the web corpus. How does detection AUC change?
6. Use the paired bootstrap to compare the two DPO/RLHF policies from L10 on simulated per-prompt rewards.

### References
* Dubois et al. 2024, *Length-Controlled AlpacaEval*; Dubois et al. 2023, *AlpacaFarm*; Zheng et al. 2023 (MT-Bench, Chatbot Arena).
* Gururangan et al. 2018, *Annotation Artifacts in Natural Language Inference Data*.
* Lin 2004 (ROUGE); Zhang et al. 2020 (BERTScore); Sellam et al. 2020 (BLEURT).
* Celikyilmaz, Clark & Gao 2020, *Evaluation of Text Generation: A Survey*; Belz et al. 2023 (reproducibility of human evaluation).
* Chen et al. 2021 (Codex, HumanEval, unbiased pass@k).
* Liang et al. 2022 (HELM); Hendrycks et al. 2021 (MMLU); Alzahrani et al. 2024 (benchmark sensitivity).
* Shi et al. 2023 (Min-K% prob); Oren et al. 2023 (proving test-set contamination).
* Santurkar et al. 2023 (OpinionQA); Marie et al. 2021 (MT evaluation practices).
* Dror et al. 2018, *The Hitchhiker's Guide to Testing Statistical Significance in NLP*; Ruder, *Challenges and Opportunities in NLP Benchmarking* (blog).

**Next:** Lecture 12, efficient neural network training.